In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "5,6"

In [8]:
demo = "请抽取原文中的所有实体，如无任何实体，请输出无\n\n### 原文:\n购销合同签约地点:上海合同编号:2020-SZHF-TAC0901签约时间:2020年 09月 01 日需方(甲方):海安翰飞铜业科技有限公司江苏省海安市城东镇晓星大道8号供方(乙方):上海顺昭国际贸易有限公司中国(上海)自由贸易试验区富特南路 301 号2幢2层 216-1 室为了保障甲乙双方合法权益,根据《中华人民共和国合同法》经协商一致同意签订本合同,双方应严格履行。一、产品明细: 注:本合同项下,最终确认溢短装不得超过 0.5%。二、付款条款:合同有效期内(自合同签订日起 180 日)付款,付款方式以人民币信用证、银行承兑汇票以及电汇等方式结算。 三、交货条款:买方自提。 四、交货期限条款:合同生效后 180 日内。五、 产品质量标准条款:按国标 GB/T467-2010 及生产厂家提供的质保单为准。电解铜应符合上海期货交易所或伦敦金属交易所电解铜交割标准。 六、违约责任条款:  甲乙双方应严格遵守、执行本合同。如一方违约,应按以下方式对守约方承担赔偿责任;1、甲方未按合同约定期限付款的(质量原因除外),每逾期一天应按货物总价的 0.01 %向乙方支付违约金,累计超过 10 个工作日乙方有权利向甲方提请诉讼。 2、乙方未按合同约定期限交货的(含质量原因导致延期交货的),每逾期一天应按货物总价的 0.01%向甲方支付违约金,还应对甲方由此造成的生产延误进行赔偿。3、乙方应对产品与合同要求不符或质量缺陷承担全部责任:3.1 产品检验以甲方的检验结论为准。 3.2 产品存在与合同要求不符或质量缺陷的,甲方有权选择拒收、退货、要求减价、更换、修理等。对于甲方生产过程中因产品质量而造成损失的,乙方应赔付甲方的相关损失,包括:(1)人工费(按甲方支付员工的实际工资计算);(2)相关材料费(按当时材料进价计算),及其他相关损失。3.3  乙方依甲方要求承担 3.2 项责任后,仍应对因产品与合同要求不符或质量缺陷而给甲方造成的经济损失承担赔偿责任。 4、产品经甲方验收不合格的,乙方应负责将货物运回(乙方可以委托甲方办理退货事宜,相关费用和风险由乙方承担);对不合格产品由乙方负责换货并在十日内(自不合格通知书发出之日起)重新发货至甲方,若经甲方验收产品依然达不到本合同约定的质量标准的,乙方应在收到甲方不合格通知书之日起三个工作日内将货款全部退回甲方,逾期不退款的需向甲方支付总货款_10 %的违约金。 七、不可抗力条款: 甲乙双方的任何一方由于不可抗力的原因不能履行合同时,应及时向对方通报不能履行或不能完全履行 的原因,以减轻可能给对方造成的损失,经对方确认后,允许延期履行,部分履行或不履行合同,并根据情 况部分或全部免于承担违约责任。 八、争议解决条款:双方本着友好合作的态度,合同履行过程中发生的违约行为进行及时的协商解决,如不能协圈解决可通过法律诉讼解决。对于双方的任何争议,经友好协商仍不能解决的,向合同签订地人民法院提请仲裁诉讼解决。九、其他条款:本合同一式二份,甲乙双方各执一份,具有同等法律效力,自双方签字盖章之日起生效,传真件与原件具有同等法律效力。双方如对合同条款有异议,经双方协商后可另行约定或签订补充协议,补充协议与本合同具有同等法律效力。 甲方(需方):(签章)乙方(供方):(签章) 星仿g授权签字人:授权签字人: \n\n### 实体类别:\n收款账户开户行，包装方式-标题，运输方式，甲方持有份数，收款账户主体名称，合同编号，甲方法定代表人/委托代理人，验收条件，乙方名称，付款时间要求，质量保证或质量要求-标题，争议解决-标题"
demo = f"""
<|im_start|>user
{demo}
<|im_end|>
<|im_start|>assistant
"""

template = """
<|im_start|>user
请抽取原文中的所有实体，如无任何实体，请输出无\n\n### 原文:\n{context} \n\n### 实体类别:\n{field_list}
<|im_end|>
<|im_start|>assistant
"""



In [3]:
# load_qwen3_32b_with_lora_fp16.py
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE = "/ldata/Qwen/Qwen3-32B"
LORA = "/ldata/share_data/llm_risk_predicting/models/llm/lora_models/checkpoint-1200"

# tokenizer（Qwen 系列通常需要 trust_remote_code=True）
tokenizer = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)

# 加载 base model（使用 device_map="auto" 让 transformers 自动把权重分配到可用设备）
model = AutoModelForCausalLM.from_pretrained(
    BASE,
    torch_dtype=torch.float16,   # fp16 推理
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True
)

# 把 LoRA adapter 挂载到 base model 上
model = PeftModel.from_pretrained(model, LORA, device_map="auto", trust_remote_code=True)

model.eval()

# 测试生成
inputs = tokenizer(demo, return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=True))


Loading checkpoint shards:   0%|          | 0/17 [00:00<?, ?it/s]

/home/gerald.hu/.conda/envs/llamafactory/lib/python3.10/site-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


请抽取原文中的所有实体，如无任何实体，请输出无

### 原文:
购销合同签约地点:上海合同编号:2020-SZHF-TAC0901签约时间:2020年 09月 01 日需方(甲方):海安翰飞铜业科技有限公司江苏省海安市城东镇晓星大道8号供方(乙方):上海顺昭国际贸易有限公司中国(上海)自由贸易试验区富特南路 301 号2幢2层 216-1 室为了保障甲乙双方合法权益,根据《中华人民共和国合同法》经协商一致同意签订本合同,双方应严格履行。一、产品明细: 注:本合同项下,最终确认溢短装不得超过 0.5%。二、付款条款:合同有效期内(自合同签订日起 180 日)付款,付款方式以人民币信用证、银行承兑汇票以及电汇等方式结算。 三、交货条款:买方自提。 四、交货期限条款:合同生效后 180 日内。五、 产品质量标准条款:按国标 GB/T467-2010 及生产厂家提供的质保单为准。电解铜应符合上海期货交易所或伦敦金属交易所电解铜交割标准。 六、违约责任条款:  甲乙双方应严格遵守、执行本合同。如一方违约,应按以下方式对守约方承担赔偿责任;1、甲方未按合同约定期限付款的(质量原因除外),每逾期一天应按货物总价的 0.01 %向乙方支付违约金,累计超过 10 个工作日乙方有权利向甲方提请诉讼。 2、乙方未按合同约定期限交货的(含质量原因导致延期交货的),每逾期一天应按货物总价的 0.01%向甲方支付违约金,还应对甲方由此造成的生产延误进行赔偿。3、乙方应对产品与合同要求不符或质量缺陷承担全部责任:3.1 产品检验以甲方的检验结论为准。 3.2 产品存在与合同要求不符或质量缺陷的,甲方有权选择拒收、退货、要求减价、更换、修理等。对于甲方生产过程中因产品质量而造成损失的,乙方应赔付甲方的相关损失,包括:(1)人工费(按甲方支付员工的实际工资计算);(2)相关材料费(按当时材料进价计算),及其他相关损失。3.3  乙方依甲方要求承担 3.2 项责任后,仍应对因产品与合同要求不符或质量缺陷而给甲方造成的经济损失承担赔偿责任。 4、产品经甲方验收不合格的,乙方应负责将货物运回(乙方可以委托甲方办理退货事宜,相关费用和风险由乙方承担);对不合格产品由乙方负责换货并在十日内(自不合格通知书发出之日起)重新发货至甲方,若经甲方验收产品依然达不到本合同约定的质量标准的,乙方应在收到甲方不合格通知

In [14]:
def decode(text, model=model):
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    result = tokenizer.decode(out[0], skip_special_tokens=True)
    return result
    

In [16]:
fields = [
    "标题",
    "地点",
    "时间",
    "角色",
    "职业",
    "引发事件",
    "关键物件",
    "情绪状态",
    "结局",
    "主题",
    "氛围",
    "字数",
]


text = template.format(
    context="冬日清晨，林夕推开面包房的玻璃门，温热的香气和新出炉的法棍把小镇唤醒。钟表匠周墨在门口修理老钟，敲开的节拍像心跳。小艾背着吉他，街角弹起一首温柔的歌，阿强骑着电车递来一封信——是多年未见的父亲写来的道歉信。四人在面包店的窗前慢慢读着，沉默被香气和音乐柔化。那天，和解像面包一样被切成片，分给每个人暖暖的一口。夕阳把面包房的窗台染成金色，林夕笑了，周墨的眼里亮起久违的光，小艾的歌声里有泪，阿强终于放下了沉重的负担。", 
    field_list="，".join(fields))
res = decode(text=text)
print(res)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



user
请抽取原文中的所有实体，如无任何实体，请输出无

### 原文:
冬日清晨，林夕推开面包房的玻璃门，温热的香气和新出炉的法棍把小镇唤醒。钟表匠周墨在门口修理老钟，敲开的节拍像心跳。小艾背着吉他，街角弹起一首温柔的歌，阿强骑着电车递来一封信——是多年未见的父亲写来的道歉信。四人在面包店的窗前慢慢读着，沉默被香气和音乐柔化。那天，和解像面包一样被切成片，分给每个人暖暖的一口。夕阳把面包房的窗台染成金色，林夕笑了，周墨的眼里亮起久违的光，小艾的歌声里有泪，阿强终于放下了沉重的负担。 

### 实体类别:
标题，地点，时间，角色，职业，引发事件，关键物件，情绪状态，结局，主题，氛围，字数

assistant
{'地点': ['面包房', '街角']}
{'时间': ['冬日清晨', '那天']}
{'角色': ['林夕', '周墨', '小艾', '阿强', '父亲']}
{'职业': ['钟表匠']}
{'关键物件': ['信', '吉他', '电车']}
{'情绪状态': ['沉重', '暖暖', '亮', '有泪']}
{'结局': ['和解']}
{'主题': ['和解']}
{'氛围': ['温热', '沉默', '暖暖', '金色', '沉重']}


In [17]:
not_exist_fields = ['城市', '付款金额', '球队名称'] 
text = template.format(
    context="冬日清晨，林夕推开面包房的玻璃门，温热的香气和新出炉的法棍把小镇唤醒。钟表匠周墨在门口修理老钟，敲开的节拍像心跳。小艾背着吉他，街角弹起一首温柔的歌，阿强骑着电车递来一封信——是多年未见的父亲写来的道歉信。四人在面包店的窗前慢慢读着，沉默被香气和音乐柔化。那天，和解像面包一样被切成片，分给每个人暖暖的一口。夕阳把面包房的窗台染成金色，林夕笑了，周墨的眼里亮起久违的光，小艾的歌声里有泪，阿强终于放下了沉重的负担。", 
    field_list="，".join(not_exist_fields))
res = decode(text=text)
print(res)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



user
请抽取原文中的所有实体，如无任何实体，请输出无

### 原文:
冬日清晨，林夕推开面包房的玻璃门，温热的香气和新出炉的法棍把小镇唤醒。钟表匠周墨在门口修理老钟，敲开的节拍像心跳。小艾背着吉他，街角弹起一首温柔的歌，阿强骑着电车递来一封信——是多年未见的父亲写来的道歉信。四人在面包店的窗前慢慢读着，沉默被香气和音乐柔化。那天，和解像面包一样被切成片，分给每个人暖暖的一口。夕阳把面包房的窗台染成金色，林夕笑了，周墨的眼里亮起久违的光，小艾的歌声里有泪，阿强终于放下了沉重的负担。 

### 实体类别:
城市，付款金额，球队名称

assistant
{}
